# ETF 动量 + 风险平价策略

直接读取 `CoreData/coreData`，通过 DolphinDB `backtest` 模块运行日频回测。
不构造项目 DSL，也不导入或调用 pandas。

时序约定：对应 `run_weekly(..., -1, ...)`，每周最后一个交易日使用严格早于
当天的历史收盘价计算信号；本地按聚宽默认双边滑点 0.246% 构造下一交易日成交价。
复权因子直接读取 `CoreData` 中由 `FundAdjFactorWorker` 写入的 `adj_factor`；
行情、成交价及持仓估值统一使用复权价格，不再叠加分红送转份额调整。
策略标的、初始资金、佣金、最低手续费和开盘撮合参数沿用聚宽版本。

In [1]:
from core.database import create_session

session = create_session()

START_DATE = "2019.01.01"
END_DATE = "2026.07.26"
ENGINE_NAME = "etf_momentum_risk_parity"
INITIAL_CASH = 100_000.0

2026-07-27 17:51:44 [SUCCESS] ts_api.py | <module>: 187行| Tushare Pro 初始化完成，共加载 5,871 只股票
2026-07-27 17:51:44 [ INFO  ] session.py | <module>: 10行| DolphinDB: 127.0.0.1:8848


## 1. 从 CoreData 构造 Backtest 日频消息

In [2]:
data_script = """
loadedPlugins = exec plugin from getLoadedPlugins()
if (!("MatchingEngineSimulator" in loadedPlugins)) {
    loadPlugin("MatchingEngineSimulator")
}
if (!("Backtest" in loadedPlugins)) {
    loadPlugin("Backtest")
}
use backtest

strategyStartDate = __START_DATE__
strategyEndDate = __END_DATE__
strategyCodes = symbol([
    "518880.SH",
    "159980.SZ",
    "159981.SZ",
    "159985.SZ",
    "501018.SH",
    "513400.SH",
    "513100.SH",
    "513500.SH",
    "513180.SH",
    "513120.SH",
    "513070.SH",
    "588000.SH",
    "159967.SZ",
    "512890.SH",
    "159851.SZ"
])
strategySymbols = symbol(
    strReplace(
        strReplace(string(strategyCodes), ".SZ", ".XSHE"),
        ".SH",
        ".XSHG"
    )
)
strategyFactors = symbol([
    "open",
    "high",
    "low",
    "close",
    "pre_close",
    "adj_factor",
    "vol"
])

strategyLongData = select
    time,
    code,
    factor,
    value
from loadTable("dfs://CoreData", "coreData")
where
    time >= temporalAdd(strategyStartDate, -120, "d"),
    time <= strategyEndDate,
    code in strategyCodes,
    factor in strategyFactors

strategyWideData = select first(value) as value
from strategyLongData
pivot by time, code, factor

strategyMarketData = select
    time,
    code,
    double(open * adj_factor) as open,
    double(low * adj_factor) as low,
    double(high * adj_factor) as high,
    double(close * adj_factor) as close,
    long(round(vol * 100, 0)) as volume,
    double(
        iif(
            high > round(pre_close * 1.1, 3),
            high,
            round(pre_close * 1.1, 3)
        ) * adj_factor
    ) as upLimitPrice,
    double(
        iif(
            low < round(pre_close * 0.9, 3),
            low,
            round(pre_close * 0.9, 3)
        ) * adj_factor
    ) as downLimitPrice,
    double(pre_close * adj_factor) as prevClosePrice
from strategyWideData
where
    !isNull(open),
    !isNull(low),
    !isNull(high),
    !isNull(close),
    !isNull(pre_close),
    !isNull(adj_factor),
    !isNull(vol),
    close > 0,
    vol >= 0
order by time, code

strategyMessageSource = select *
from strategyMarketData
where
    date(time) >= strategyStartDate,
    date(time) <= strategyEndDate

strategyMessage = backtest::build_backtest_message(
    strategyMessageSource
)
strategyCalendar = select distinct
    date(tradeTime) as tradeDate
from strategyMessage
strategyCalendar = select
    tradeDate,
    next(tradeDate) as nextTradeDate
from strategyCalendar
order by tradeDate
strategyRebalanceDates = exec tradeDate
from strategyCalendar
where
    isNull(nextTradeDate) ||
    weekOfYear(nextTradeDate) != weekOfYear(tradeDate)

profile = dict(STRING, ANY)
profile["sourceRows"] = strategyLongData.rows()
profile["marketRows"] = strategyMarketData.rows()
profile["messageRows"] = strategyMessage.rows()
profile["adjustedFactorCodes"] = size(
    exec distinct code
    from strategyWideData
    where !isNull(adj_factor)
)
profile["missingAdjFactorRows"] = exec count(*)
from strategyWideData
where
    date(time) >= strategyStartDate,
    date(time) <= strategyEndDate,
    !isNull(close),
    isNull(adj_factor)
profile["firstMessageTime"] = min(strategyMessage.tradeTime)
profile["lastMessageTime"] = max(strategyMessage.tradeTime)
profile["rebalanceDates"] = size(strategyRebalanceDates)
profile
"""

data_profile = session.run(
    data_script
    .replace("__START_DATE__", START_DATE)
    .replace("__END_DATE__", END_DATE)
)
data_profile

{'messageRows': 22191,
 'lastMessageTime': np.datetime64('2026-07-24T15:00:00.000'),
 'rebalanceDates': 387,
 'firstMessageTime': np.datetime64('2019-01-02T15:00:00.000'),
 'missingAdjFactorRows': 0,
 'adjustedFactorCodes': 15,
 'marketRows': 22507,
 'sourceRows': 157554}

## 2. 定义策略并运行回测

In [3]:
strategy_script = """
def etfMomentumScore(history, targetCode, window) {
    rows = select top __MOMENTUM_WINDOW__
        time,
        close
    from history
    where
        code == targetCode,
        !isNull(close),
        close > 0
    order by time desc
    if (rows.rows() < window) return double(NULL)

    prices = double(reverse(rows.close))
    logReturns =
        log(prices[1:window]) -
        log(prices[0:(window - 1)])
    volatility = stdp(logReturns) * sqrt(252.0)

    x = double(0..(window - 1))
    // numpy.polyfit 的 w 作用于残差，正规方程中的权重为 w²。
    regressionWeights = square(1.0 + x / (window - 1))
    y = log(prices)
    xMean = sum(regressionWeights * x) / sum(regressionWeights)
    yMean = sum(regressionWeights * y) / sum(regressionWeights)
    denominator = sum(
        regressionWeights * (x - xMean) * (x - xMean)
    )
    if (denominator <= 0) return double(NULL)

    slope = sum(
        regressionWeights *
        (x - xMean) *
        (y - yMean)
    ) / denominator
    annualReturn = exp(slope * 252.0) - 1.0
    return annualReturn / (volatility + 1e-6)
}

def etfRiskParityCovariance(history, codes, window) {
    priceHistory = select
        time,
        code,
        close
    from history
    where
        code in codes,
        !isNull(close),
        close > 0
    if (priceHistory.rows() == 0) {
        return (matrix(DOUBLE, size(codes), size(codes)), 0)
    }

    historyWide = select first(close) as close
    from priceHistory
    pivot by time, code
    historyWide = select top 90 *
    from historyWide
    order by time desc

    availableCodes = columnNames(historyWide)[1:]
    missingCodes = string(codes)[
        !(string(codes) in availableCodes)
    ]
    if (size(missingCodes) > 0) {
        return (matrix(DOUBLE, size(codes), size(codes)), 0)
    }

    priceColumns = historyWide[codes]
    validRows = take(true, historyWide.rows())
    for (index in 0..(size(codes) - 1)) {
        validRows = validRows && !isNull(priceColumns[index])
    }
    validIndices = (0..(historyWide.rows() - 1))[validRows]
    observationCount = min(size(validIndices), window)
    if (observationCount < window) {
        return (
            matrix(DOUBLE, size(codes), size(codes)),
            observationCount
        )
    }
    validIndices = reverse(validIndices[0:observationCount])

    count = size(codes)
    returns = array(ANY, 0)
    for (index in 0..(count - 1)) {
        prices = double(priceColumns[index][validIndices])
        returns.append!(
            log(prices[1:window]) -
            log(prices[0:(window - 1)])
        )
    }

    covariance = matrix(DOUBLE, count, count)
    for (leftIndex in 0..(count - 1)) {
        for (rightIndex in leftIndex..(count - 1)) {
            value = 252.0 * covar(
                returns[leftIndex],
                returns[rightIndex]
            )
            covariance[leftIndex, rightIndex] = value
            covariance[rightIndex, leftIndex] = value
        }
    }
    for (index in 0..(count - 1)) {
        covariance[index, index] += 1e-6
    }
    return (covariance, observationCount)
}

def etfRiskParityObjective(weights, covariance) {
    count = size(weights)
    covarianceTimesWeights = take(0.0, count)
    for (index in 0..(count - 1)) {
        covarianceTimesWeights[index] = sum(
            flatten(covariance[index,]) * weights
        )
    }
    portfolioVolatility = sqrt(
        sum(weights * covarianceTimesWeights)
    )
    if (portfolioVolatility <= 0) return double("inf")
    riskContributions =
        weights * covarianceTimesWeights / portfolioVolatility
    targetContribution = portfolioVolatility / count
    return sum(square(riskContributions - targetContribution))
}

def etfWeightSumConstraint(weights) {
    return sum(weights) - 1.0
}

def etfWeightSumJacobian(weights) {
    return take(1.0, size(weights))
}

def etfNonnegativeConstraint(weights) {
    return weights
}

def etfNonnegativeJacobian(weights) {
    count = size(weights)
    result = matrix(DOUBLE, count, count, 0)
    for (index in 0..(count - 1)) result[index, index] = 1.0
    return result
}

def etfRiskParityWeights(covariance, maxIterations, tolerance) {
    count = rows(covariance)
    if (count == 0 || cols(covariance) != count) {
        throw "协方差矩阵必须是非空方阵"
    }

    equalityConstraint = dict(STRING, ANY)
    equalityConstraint[`type] = `eq
    equalityConstraint[`fun] = etfWeightSumConstraint
    equalityConstraint[`jac] = etfWeightSumJacobian
    nonnegativeConstraint = dict(STRING, ANY)
    nonnegativeConstraint[`type] = `ineq
    nonnegativeConstraint[`fun] = etfNonnegativeConstraint
    nonnegativeConstraint[`jac] = etfNonnegativeJacobian
    bounds = matrix(take(0.0, count), take(1.0, count))
    optimization = fminSLSQP(
        etfRiskParityObjective{, covariance},
        take(1.0 / count, count),
        constraints=[equalityConstraint, nonnegativeConstraint],
        bounds=bounds,
        ftol=tolerance,
        maxIter=maxIterations
    )
    if (optimization[`mode] != 0) {
        throw "SLSQP 风险平价优化失败"
    }
    return optimization[`xopt]
}

def etfInitialize(mutable context) {
    context["rebalanceCount"] = 0
    context["rebalanceDateLog"] = array(DATE, 0)
    context["selectionLog"] = array(STRING, 0)
    context["weightLog"] = array(STRING, 0)
    context["portfolioVolatilityLog"] = array(DOUBLE, 0)
    context["targetExposureLog"] = array(DOUBLE, 0)
}

def etfOnBar(mutable context, msg, indicator) {
    if (msg.rows() == 0) return
    currentDate = date(msg.tradeTime[0])
    slippage = 0.00246 / 2.0
    pricePrecision = 3
    messageSymbols = symbol(string(msg.symbol))
    messageVolumes = long(msg.volume)
    calendarData = context["coreBacktestUnfilteredFactorData"]
    nextTradeDate = exec min(date(time))
    from calendarData
    where date(time) > currentDate
    if (
        !isNull(nextTradeDate) &&
        weekOfYear(nextTradeDate) == weekOfYear(currentDate)
    ) return

    strategyCodes = symbol([
        "518880.SH",
        "159980.SZ",
        "159981.SZ",
        "159985.SZ",
        "501018.SH",
        "513400.SH",
        "513100.SH",
        "513500.SH",
        "513180.SH",
        "513120.SH",
        "513070.SH",
        "588000.SH",
        "159967.SZ",
        "512890.SH",
        "159851.SZ"
    ])
    strategySymbols = symbol(
        strReplace(
            strReplace(string(strategyCodes), ".SZ", ".XSHE"),
            ".SH",
            ".XSHG"
        )
    )

    // 只读取当前消息日期之前的数据，禁止使用当日收盘价。
    history = backtest::getHistoryData(context, msg, false)
    if (history.rows() == 0) return

    scores = take(double(NULL), size(strategyCodes))
    for (index in 0..(size(strategyCodes) - 1)) {
        scores[index] = etfMomentumScore(
            history,
            strategyCodes[index],
            __MOMENTUM_WINDOW__
        )
    }
    scoreTable = table(strategyCodes as code, scores as score)
    selected = select top __SELECT_COUNT__ *
    from scoreTable
    where !isNull(score)
    order by score desc
    if (selected.rows() < __SELECT_COUNT__) return

    selectedCodes = selected.code
    selectedSymbols = symbol(
        strReplace(
            strReplace(string(selectedCodes), ".SZ", ".XSHE"),
            ".SH",
            ".XSHG"
        )
    )
    covarianceResult = etfRiskParityCovariance(
        history,
        selectedCodes,
        __RISK_WINDOW__
    )
    if (covarianceResult[1] < __RISK_WINDOW__) return

    covariance = covarianceResult[0]
    weights = etfRiskParityWeights(covariance, 1000, 1e-9)
    marginalRisk = take(0.0, size(weights))
    for (index in 0..(size(weights) - 1)) {
        marginalRisk[index] = sum(
            flatten(covariance[index,]) * weights
        )
    }
    portfolioVolatility = sqrt(sum(weights * marginalRisk))
    targetExposure = 1.0
    if (portfolioVolatility > __TARGET_VOLATILITY__) {
        targetExposure =
            __TARGET_VOLATILITY__ / portfolioVolatility
        weights *= targetExposure
    }

    // 聚宽在 09:30 回调中按当日开盘价估值可交易持仓；停牌持仓
    // 没有当日行情，继续使用最近一个收盘价估值。
    totalEquity = Backtest::getAvailableCash(context.engine)
    pausedValue = 0.0
    for (index in 0..(size(strategySymbols) - 1)) {
        currentSymbol = strategySymbols[index]
        position = Backtest::getPosition(
            context.engine,
            currentSymbol
        )
        currentPosition = long(
            nullFill(position.longPosition.sum(), 0)
        )
        messageIndex = find(messageSymbols, currentSymbol)
        paused = messageIndex < 0
        if (!paused) paused = messageVolumes[messageIndex] <= 0
        if (currentPosition > 0 && !paused) {
            totalEquity += currentPosition * msg.open[messageIndex]
        }
        if (currentPosition > 0 && paused) {
            lastPrice = select top 1 close
            from history
            where code == strategyCodes[index]
            order by time desc
            if (lastPrice.rows() > 0) {
                positionValue = currentPosition * lastPrice.close[0]
                pausedValue += positionValue
                totalEquity += positionValue
            }
        }
    }
    if (isNull(totalEquity) || totalEquity <= 0) return

    allocatableEquity = 0.99 * (totalEquity - pausedValue)
    if (allocatableEquity <= 0) return
    targetPositions = take(long(0), size(selectedSymbols))
    executionPrices = take(double(NULL), size(selectedSymbols))
    for (index in 0..(size(selectedSymbols) - 1)) {
        currentSymbol = selectedSymbols[index]
        messageIndex = find(messageSymbols, currentSymbol)
        position = Backtest::getPosition(
            context.engine,
            currentSymbol
        )
        currentPosition = long(
            nullFill(position.longPosition.sum(), 0)
        )
        paused = messageIndex < 0
        if (!paused) paused = messageVolumes[messageIndex] <= 0
        if (paused) {
            targetPositions[index] = currentPosition
        } else {
            executionPrices[index] = msg.open[messageIndex]
            targetValue = allocatableEquity * weights[index]
            positionDifference =
                targetValue / executionPrices[index] - currentPosition
            positionAdjustment = long(
                floor(abs(positionDifference) / 100.0) * 100
            )
            targetPositions[index] = iif(
                positionDifference < 0,
                currentPosition - positionAdjustment,
                currentPosition + positionAdjustment
            )
        }
    }

    // 先卖出被剔除或超配的基金。
    for (index in 0..(size(strategySymbols) - 1)) {
        currentSymbol = strategySymbols[index]
        messageIndex = find(messageSymbols, currentSymbol)
        if (
            messageIndex < 0 ||
            messageVolumes[messageIndex] <= 0
        ) continue

        position = Backtest::getPosition(
            context.engine,
            currentSymbol
        )
        currentPosition = long(
            nullFill(position.longPosition.sum(), 0)
        )
        selectedIndex = find(selectedSymbols, currentSymbol)
        targetPosition = iif(
            selectedIndex < 0,
            long(0),
            targetPositions[selectedIndex]
        )
        if (currentPosition > targetPosition) {
            Backtest::submitOrder(
                context.engine,
                (
                    currentSymbol,
                    context.tradeTime,
                    5,
                    round(
                        msg.open[messageIndex] * (1.0 - slippage),
                        pricePrecision
                    ),
                    currentPosition - targetPosition,
                    3
                ),
                "etf-risk-parity-sell"
            )
        }
    }

    // 再买入低配基金。
    for (index in 0..(size(selectedSymbols) - 1)) {
        currentSymbol = selectedSymbols[index]
        messageIndex = find(messageSymbols, currentSymbol)
        if (
            messageIndex < 0 ||
            messageVolumes[messageIndex] <= 0
        ) continue

        position = Backtest::getPosition(
            context.engine,
            currentSymbol
        )
        currentPosition = long(
            nullFill(position.longPosition.sum(), 0)
        )
        targetPosition = targetPositions[index]
        if (currentPosition <= targetPosition - 100) {
            Backtest::submitOrder(
                context.engine,
                (
                    currentSymbol,
                    context.tradeTime,
                    5,
                    round(
                        executionPrices[index] * (1.0 + slippage),
                        pricePrecision
                    ),
                    targetPosition - currentPosition,
                    1
                ),
                "etf-risk-parity-buy"
            )
        }
    }

    context["rebalanceCount"] += 1
    context["rebalanceDateLog"].append!(currentDate)
    selectionText = concat(string(selectedCodes), ",")
    weightText = concat(string(round(weights, 6)), ",")
    context["selectionLog"].append!(selectionText)
    context["weightLog"].append!(weightText)
    context["portfolioVolatilityLog"].append!(portfolioVolatility)
    context["targetExposureLog"].append!(targetExposure)
}

def etfOnOrder(mutable context, orders) {
}

def etfOnTrade(mutable context, trades) {
}

strategyConfig = dict(STRING, ANY)
strategyConfig["startDate"] = strategyStartDate
strategyConfig["endDate"] = strategyEndDate
strategyConfig["strategyGroup"] = "stock"
strategyConfig["cash"] = double(__INITIAL_CASH__)
strategyConfig["commission"] = double(0.00005)
strategyConfig["enableMinimumPerTransactionFee"] = true
strategyConfig["tax"] = double(0)
strategyConfig["dataType"] = int(4)
strategyConfig["msgAsTable"] = true
strategyConfig["matchingMode"] = int(3)

strategyEngine = backtest::run_backtest(
    "__ENGINE_NAME__",
    strategyConfig,
    strategyMessage,
    strategyMarketData,
    strategyMarketData,
    etfInitialize,
    NULL,
    etfOnBar,
    NULL,
    etfOnOrder,
    etfOnTrade,
    NULL,
    NULL
)

strategyContext = Backtest::getContextDict(strategyEngine)
strategyPortfolios = Backtest::getDailyTotalPortfolios(
    strategyEngine
)
strategyTrades = Backtest::getTradeDetails(strategyEngine)
strategyReturnSummary = backtest::standardize_return_summary(
    Backtest::getReturnSummary(strategyEngine),
    strategyPortfolios,
    250,
    0.04
)
strategyEngineStat = Backtest::getBacktestEngineStat(
    strategyEngine
)

runResult = dict(STRING, ANY)
runResult["status"] = strategyEngineStat.status[0]
runResult["lastError"] = strategyEngineStat.lastErrMsg[0]
runResult["messageRows"] = strategyMessage.rows()
runResult["rebalanceCount"] = strategyContext["rebalanceCount"]
runResult["orderStatusRows"] = strategyTrades.rows()
runResult["portfolioDays"] = strategyPortfolios.rows()
if (strategyPortfolios.rows() > 0) {
    runResult["finalEquity"] = last(strategyPortfolios.totalEquity)
}
summaryColumns = columnNames(strategyReturnSummary)
if ("totalReturn" in summaryColumns) {
    runResult["totalReturn"] = strategyReturnSummary.totalReturn[0]
}
if ("annualReturn" in summaryColumns) {
    runResult["annualReturn"] = strategyReturnSummary.annualReturn[0]
}
if ("annualVolatility" in summaryColumns) {
    runResult["annualVolatility"] = strategyReturnSummary.annualVolatility[0]
}
if ("sharpeRatio" in summaryColumns) {
    runResult["sharpeRatio"] = strategyReturnSummary.sharpeRatio[0]
}
if ("maxDrawdown" in summaryColumns) {
    runResult["maxDrawdown"] = strategyReturnSummary.maxDrawdown[0]
}
runResult
"""

strategy_result = session.run(
    strategy_script
    .replace("__ENGINE_NAME__", ENGINE_NAME)
    .replace("__INITIAL_CASH__", str(float(INITIAL_CASH)))
    .replace("__MOMENTUM_WINDOW__", "21")
    .replace("__SELECT_COUNT__", "4")
    .replace("__RISK_WINDOW__", "20")
    .replace("__TARGET_VOLATILITY__", "0.09")
)
strategy_result

{'maxDrawdown': 0.09219227107663565,
 'sharpeRatio': 1.2658959200128168,
 'lastError': None,
 'rebalanceCount': 386,
 'totalReturn': 2.388333950300001,
 'portfolioDays': 1833,
 'annualReturn': 0.18109271014717754,
 'orderStatusRows': 3818,
 'messageRows': 22191,
 'annualVolatility': 0.11145680139781872,
 'status': 'END',
 'finalEquity': 338833.39503000013}

## 3. 查看最近一次调仓

In [4]:
last_rebalance = session.run("""
result = dict(STRING, ANY)
result["date"] = last(strategyContext["rebalanceDateLog"])
result["selected"] = last(strategyContext["selectionLog"])
result["weights"] = last(strategyContext["weightLog"])
result["annualizedVolatility"] = last(
    strategyContext["portfolioVolatilityLog"]
)
result["targetExposure"] = last(
    strategyContext["targetExposureLog"]
)
result
""")
last_rebalance

{'targetExposure': 0.411356723608117,
 'annualizedVolatility': 0.2187882070106611,
 'weights': '0.083749,0.0521,0.164814,0.110694',
 'selected': '513120.SH,501018.SH,159985.SZ,513180.SH',
 'date': np.datetime64('2026-07-17')}

## 4. 清理回测引擎

In [5]:
session.run(
    "Backtest::dropBacktestEngine(strategyEngine)"
)
session.close()